# Pricing Engine Benchmarks
This notebook provides a comprehensive benchmark of different pricing models (Black-Scholes Pseudo Random, Black-Scholes Sobol, and Heston) across a variety of financial products. 
The products range from standard Vanilla options to complex Path-Dependent, Early-Exercise, and Structured Autocall products.

In [1]:
import time
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..'))
pd.options.display.float_format = '{:,.4f}'.format

from kernel.market_data.data_loader import MarketDataLoader
from kernel.market_data.market import Market
from kernel.market_data.volatility_surface.enums_volatility import VolatilitySurfaceType
from kernel.market_data.rate_curve_data.enums_interpolators import InterpolationType
from kernel.tools import CalendarConvention, ObservationFrequency, RateCurveType, Model
from utils.pricing_settings import PricingSettings
from kernel.pricing_launcher import PricingLauncher
from kernel.models.pricing_engines.enum_pricing_engine import PricingEngineType

# Products
from kernel.products.options.vanilla_options import EuropeanCallOption, EuropeanPutOption
from kernel.products.options.barrier_options import DownAndInCallOption, UpAndOutPutOption
from kernel.products.options.binary_options import BinaryCallOption
from kernel.products.options.path_dependent_options import AsianCallOption, LookbackPutOption
from kernel.products.options_strategies.options_strategies import Straddle, BullSpread
from kernel.products.options.american_options import AmericanCallOption, BermudanPutOption
from kernel.products.structured_products.autocall_products import Phoenix, Eagle


## 1. Initialize Market Data
We start by loading SPX data and calibrating the underlying rates and volatility surfaces (SVI).

In [2]:
print("Initializing Market Data...")
t0 = time.time()
settings = PricingSettings()
settings.underlying_name = "SPX"
settings.rate_curve_type = RateCurveType.RF_US_TREASURY
settings.interpolation_type = InterpolationType.CUBIC
settings.day_count_convention = CalendarConvention.ACT_360
settings.obs_frequency = ObservationFrequency.ANNUAL
settings.random_generator_type = "SOBOL"
settings.model = Model.BLACK_SCHOLES
settings.volatility_surface_type = VolatilitySurfaceType.SVI
settings.compute_greeks = True
settings.nb_paths = 10000  
settings.nb_steps = 256

data_loader = MarketDataLoader()
underlying_df = data_loader.get_underlying_info(settings.underlying_name)
options_df = data_loader.get_option_data(settings.underlying_name)
yield_df = data_loader.get_yield_curve(settings.rate_curve_type.value)

market = Market(
    underlying_name=settings.underlying_name,
    yield_curve_data=yield_df,
    underlying_data=underlying_df,
    option_data=options_df,
    rate_curve_type=settings.rate_curve_type,
    interpolation_type=settings.interpolation_type, 
    volatility_surface_type=settings.volatility_surface_type,
    calendar_convention=settings.day_count_convention,
    obs_frequency=settings.obs_frequency
)

spot = market.underlying_asset.last_price
print(f"Market init time: {time.time() - t0:.4f} seconds")
print(f"Current Underlying Spot Price: {spot:.2f}")


Initializing Market Data...
Market init time: 0.3500 seconds
Current Underlying Spot Price: 5768.00


## 2. Define Portfolios
We group our test assets into four portfolios: European, Path-Dependent, American (Early-Exercise), and Structured Products.

In [3]:
european_products = [
    ("European Call", EuropeanCallOption(strike=spot, maturity=1.0)),
    ("European Put", EuropeanPutOption(strike=spot*0.9, maturity=1.0)),
    ("Cash-Or-Nothing Call", BinaryCallOption(strike=spot, maturity=1.0, coupon=100)),
    ("Straddle", Straddle(strike=spot, maturity=1.0, position_call=True, position_put=True)),
    ("Bull Spread", BullSpread(maturity=1.0, strike_low=spot*0.9, strike_high=spot*1.1, position_low=True, position_high=False)),
]

path_dependent_products = [
    ("Down-And-In Call", DownAndInCallOption(strike=spot, maturity=1.0, barrier=spot*0.8)),
    ("Up-And-Out Put", UpAndOutPutOption(strike=spot, maturity=1.0, barrier=spot*1.2)),
    ("Asian Call", AsianCallOption(strike=spot, maturity=1.0)),
    ("Lookback Put", LookbackPutOption(strike=spot, maturity=1.0)),
]

american_products = [
    ("American Call", AmericanCallOption(strike=spot, maturity=1.0)),
    ("Bermudan Put", BermudanPutOption(strike=spot, maturity=1.0, exercise_times=[0.25, 0.5, 0.75, 1.0])),
]

structured_products = [
    ("Phoenix Autocall", Phoenix(maturity=3.0, observation_frequency=ObservationFrequency.SEMIANNUAL, capital_barrier=70.0, autocall_barrier=100.0, coupon_barrier=80.0, coupon_rate=8.0)),
    ("Eagle Autocall", Eagle(maturity=3.0, observation_frequency=ObservationFrequency.SEMIANNUAL, capital_barrier=70.0, autocall_barrier=100.0, coupon_rate=8.0)),
]


## 3. Pricing Execution Engine
We create a helper function that runs the benchmark configurations across the portfolios and styles the pandas DataFrames.

In [4]:
launcher = PricingLauncher(pricing_settings=settings, market=market)

def run_batch(products, engine_type):
    results = []
    configs = [
        {"name": "BS (Pseudo)", "model": Model.BLACK_SCHOLES, "gen": "NUMPY", "paths": 10000},
        {"name": "BS (Sobol)", "model": Model.BLACK_SCHOLES, "gen": "SOBOL", "paths": 5000},
        {"name": "Heston", "model": Model.HESTON, "gen": "NUMPY", "paths": 10000}
    ]
    
    for config in configs:
        settings.pricing_engine_type = engine_type
        settings.model = config["model"]
        settings.random_generator_type = config["gen"]
        settings.nb_paths = config["paths"]
        
        for name, prod in products:
            start = time.time()
            res = launcher.calculate(prod)
            elapsed = time.time() - start
            
            results.append({
                "Product": name,
                "Model": config["name"],
                "Price": res.price,
                "Time(s)": elapsed,
                "StdDev": res.std_dev if (res.std_dev is not None and config["gen"] != "SOBOL") else pd.NA,
                "Delta": res.greeks.get('delta', pd.NA) if res.greeks else pd.NA,
                "Gamma": res.greeks.get('gamma', pd.NA) if res.greeks else pd.NA,
                "Vega": res.greeks.get('vega', pd.NA) if res.greeks else pd.NA
            })
            
    df = pd.DataFrame(results)
    return display_results(df)

from IPython.display import display

def display_results(df):
    pivot_df = df.pivot(index='Product', columns='Model', values=['Price', 'Time(s)', 'StdDev', 'Delta', 'Gamma', 'Vega'])
    pivot_df = pivot_df.reindex(columns=["BS (Pseudo)", "BS (Sobol)", "Heston"], level=1)
    
    # Styling
    styler = pivot_df.style.format(na_rep="N/A", precision=4)
    styler = styler.background_gradient(subset=pd.IndexSlice[:, (['Price', 'Time(s)'], slice(None))], cmap="Blues")
    
    display(styler)


## 4. Benchmarks
### European Products
Standard pricing engine tests.

In [5]:
run_batch(european_products, PricingEngineType.MC)


### Path Dependent Products
Tests utilizing continuous path evaluation (Asians, Lookbacks, Barriers).

In [6]:
run_batch(path_dependent_products, PricingEngineType.MC)


### Early Exercise (American/Bermudan)
Tests utilizing American Monte Carlo (Longstaff-Schwartz regression techniques).

In [7]:
run_batch(american_products, PricingEngineType.AMERICAN_MC)


### Structured Products
Tests utilizing complex callable structures.

In [8]:
run_batch(structured_products, PricingEngineType.CALLABLE_MC)
